# 4.1 Meshes Versus Generators

FrequenSolve can consume a supplied mesh or generate one from model geometry. Generated meshes are recommended for most layered workflows because the generator preserves geometric information for later adaptivity and geometry-aware refinement.

By the end, you should be able to decide when to use a generated mesh versus a supplied GMP mesh, and inspect the generated mesh payload before adaptivity.


## How To Read This Tutorial

Meshing in FrequenSolve is intentionally generator-first. A provided mesh is possible, but a generator preserves model geometry and lets the solver/adaptivity machinery reason about interfaces, PML extrusion, and later refinement.

Read this notebook as a comparison of ownership: the model owns geometry and properties, the mesh generator owns how an initial discretization is made, and the job/output request owns what mesh artifacts are written for inspection.

## Design Notes

Generated meshes are the preferred starting point when the model geometry is available in Python. They preserve surfaces, layer labels, model bounds, and PML context so later adaptivity can reason about geometry. Supplied meshes are still useful when another tool owns the geometry, but the current supported external path is the FrequenSolve/GMP mesh format.

| Choice | What you provide | What the solver can still infer |
| --- | --- | --- |
| Mesh generator | Bounds, counts, surfaces, layer geometry | Geometry-aware refinement, boundary labels, PML sides, region labels. |
| Supplied GMP mesh | Initial topology and geometry records | Whatever labels and geometry are present in the GMP file. |
| Runtime adaptive mesh | Produced by the solver from the initial mesh | Wavefield- and geometry-aware refinement for the actual solve. |

Initial meshes do not need to resolve the final wavefield. In an adaptive workflow, a coarse and inspectable starting mesh is often better than hand-tuning a dense mesh too early.


## Initial Mesh Quality Versus Adaptive Mesh Quality

A generated initial mesh is a starting point, not the final numerical promise. It should preserve the geometry, labels, interfaces, and PML extrusion needed by later refinement. It does not always need ideal aspect ratios or final wavelength resolution on the first pass.

FrequenSolve supports quad, triangle, hex, tet, prism, and pyramid elements in its internal mesh path. External mesh interchange is intentionally narrow today; the recommended path is to keep geometry in the model/generator so later adaptivity can still reason about surfaces and material domains.

## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display
import frequensolve as fs

u = fs.ureg


## Model And Initial Geometry

This is a coarse starting model. Initial meshes do not need to resolve the wavefield or have final-quality aspect ratios; solver-side adaptivity can improve resolution and element quality where needed.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="mesh_vs_generators",
    path="./scratch/tutorials/mesh_vs_generators",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="mesh_vs_generators",
    physics="acoustic",
    dimension=2,
    units={
        "length": "km",
        "velocity": "km/s",
        "density": "g/cm^3",
    },
)

model = fs.LayeredModel(
    name="model",
    dimension=2,
    x_limits=[0.0, 1.0],
)
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.4 * u.km)
sim += model
model.plot("vp", figsize=(7, 3), aspect="equal")


## Frequency-Domain Mesh Inspection Run

This strict run requests ParaView output so the mesh and fields can be inspected. The current custom GMP format is the supported external mesh path; generated meshes support quad, triangle, hex, tet, prism, and pyramid element families where the solver supports them.

This job uses `upscale=0` because the goal is mesh QC, not a smoothed publication image. Native output keeps element edges and block structure easier to inspect.


In [ ]:
sim += fs.LayeredMeshGenerator(l_bound=[0.0, 0.0], u_bound=[1.0, 0.4], n=[4, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=3, f_low=5.0, f_high=20.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(
    conditions=["pml"],
    boundaries=["x_min", "x_max", "z_max"],
    pml_wavelengths=0.5,
)

acq = fs.Acquisition()
acq.add_sources(kind="scalar", coords=[[0.5, 0.05]])
node = fs.ReceiverNode(name="hydrophone")
node.add_component(name="p", field="pressure")
acq.add_receiver_group(
    name="surface",
    device=node,
    coords=[[x, 0.025] for x in np.linspace(0.1, 0.9, 21)],
)
sim += acq
sim += fs.Discretization()

site = fs.Site()
job = fs.FrequencyDomainJob(
    name="freq_mesh",
    simulation=sim,
    f_list=[20.0],
    outputs=[
        fs.VtkOutput.domain(
            name="mesh",
            fields=["pressure"],
            properties=["vp", "rho", "Subdomain"],
            show_pml=True,
            upscale=0,
            order=1,
        )
    ],
)
result = site.submit(job).wait()


## Inspect The Mesh Generator Payload

A generated mesh keeps the authoring parameters in the simulation. That is different from handing the solver an opaque mesh file: the project still knows the model bounds, initial element counts, and geometry that later adaptivity can use. External meshes are currently expected through the FrequenSolve/GMP mesh path when needed; broader mesh-format support can be added as use cases appear.


In [ ]:
mesh_payload = sim.mesh.to_fs(sim.export_context())
{
    "generator_type": mesh_payload.get("generator", {}).get("_type"),
    "initial_n": mesh_payload.get("generator", {}).get("n"),
    "adaptivity": mesh_payload.get("adapt"),
}


## List Result Files

Frequency-domain jobs always write traces; this job also requested ParaView/VTK output. `result.output_files(...)` is the stable way to discover generated files without hard-coding result paths.


In [ ]:
vtu_files = result.output_files(base="mesh", suffix=".vtu", existing=True)
all_outputs = result.output_files(existing=True)
{
    "vtu_files": [str(path) for path in vtu_files],
    "all_output_count": len(all_outputs),
}


## Persist A PyVista Screenshot

The screenshot is saved as a local asset so the rendered mesh remains visible after reopening the notebook. `show_edges=True` overlays element edges, which is the most useful view when checking whether a starting mesh is coarse, structured, or over-refined.


In [ ]:
image_dir = Path("./assets")
image_dir.mkdir(exist_ok=True)
screenshot = image_dir / "mesh_vs_generators_vp.png"
plotter = fs.plot_vtu(
    vtu_files[0],
    field="vp",
    show_edges=True,
    scalar_bar=True,
    show=False,
    screenshot=screenshot,
    window_size=(1100, 500),
)
display(Image(filename=str(screenshot)))


## Before Moving On

A coarse initial mesh is not a failure if the adaptive workflow will refine it. The review question is whether the generator captured the right geometry and produced a valid starting point, not whether the first mesh already resolves every wavelength.

Use ParaView or PyVista screenshots as persistent evidence of the mesh and property export. They make meshing decisions reviewable after the live plotting session is gone.

## Result Review Checklist

The purpose of this tutorial is to distinguish geometry ownership from mesh topology. Generated meshes are preferred because the model geometry remains available to adaptivity and output selection.

| Artifact | What to inspect |
| --- | --- |
| Mesh payload | Generator type, bounds, initial counts, and adaptivity controls are explicit. |
| ParaView edge view | The initial mesh is intentionally coarse but still spans the intended model bounds. |
| Output files | Result discovery uses `result.output_files(...)` instead of hard-coded paths. |
| Project files | The saved simulation records enough geometry to regenerate/adapt the mesh later. |

A supplied GMP mesh is appropriate when another tool owns the mesh. If the Python model owns the geometry, prefer a generator so later adaptive workflows can still reason about surfaces and regions.
